# Notebook Description Math Problem Classification with Classical Machine Learning Models

In this notebook I explored how much simple classical models are performing on the math problem classification data from
[this](https://www.kaggle.com/competitions/classification-of-math-problems-by-kasut-academy/overview) Kaggle competition.

What this Notebook has:
- First it trains Logistic Regression and LightGBM models using sklearn.
- Many samples have some gibberish text in them, so preprocessing is applied to handle that.
    - Over-preprocessing degraded scores, so only processes the most unneccessary parts of the text.
    - It is simple, but more robust preprocessing function.
    - Despite of removing a lot of gibberish, the accuracy didn't improve a lot
- Since there is class imbalance in the data, SMOTE is used to oversample the minority classes.
- With SMOTE LGBM score improved by 1 point, which is not a much of a improvement.

Next steps for this exploration:
- LGBM is overfitting and it should be hadnled properly.
- Hyperparameter Tuning can improve the score.
- Only TFIDF features are used to train the models. Features engineering (ex: counts of particular words) will help the models pick the relationships.

**Public Test Score:** 0.7862

# Install Packages

In [ ]:
!pip install imbalanced-learn -q

# Import Libraries

In [ ]:
import re
import pandas as pd
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer

# Load Data

In [ ]:
train_data_path = "https://raw.githubusercontent.com/PrudhvirajuChekuri/Final-Project-Group8/refs/heads/master/code/data/train.csv"
test_data_path = "https://raw.githubusercontent.com/PrudhvirajuChekuri/Final-Project-Group8/refs/heads/master/code/data/test.csv"
sample_sub_path = "https://raw.githubusercontent.com/PrudhvirajuChekuri/Final-Project-Group8/refs/heads/master/code/data/sample_submission.csv"

train = pd.read_csv(train_data_path)
test = pd.read_csv(test_data_path)
sample_sub = pd.read_csv(sample_sub_path)

display(train.shape, test.shape)
display(sample_sub.head())

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train["label"].value_counts()

In [ ]:
for ind, i in enumerate(train["Question"][80:100]):
    print("Q:", ind)
    print(i)
    print("\n")

- Many problems starts with random things in different forms in many examples like
    - numbers
    - Example ...
    - task ...
- Has links, random names
- Has names of question creators, authors etc
- Maybe extracted or scraped questions and are not derived by humans. Or these things are introduced randomly for competition purposes.

In [ ]:
X = train["Question"]
y = train["label"]
display(X.shape, y.shape, X.head(2), y.head(2))

# Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

# Create Training Pipeline with Logistic Regression

In [ ]:
lr = Pipeline([('vect', CountVectorizer()),
               ('tfidf', TfidfTransformer()),
               ('clf', LogisticRegression(class_weight="balanced", multi_class='ovr'))
              ])
lr.fit(X_train, y_train)

# Make Predictions

In [ ]:
train_preds_lr = lr.predict(X_train)
test_preds_lr = lr.predict(X_test)

# Evaluate

In [ ]:
print("Logistic Regression Train Accuracy:", lr.score(X_train, y_train))
print("Logistic Regression Test Accuracy:", lr.score(X_test, y_test))

# Create Training Pipeline with LightGBM

In [ ]:
lgbm = Pipeline([('vect', CountVectorizer()),
               ('tfidf', TfidfTransformer()),
               ('clf', LGBMClassifier(is_unbalance = True, verbose = -1))
              ])

lgbm.fit(X_train, y_train)

# Make Predictions

In [ ]:
train_preds_lgbm = lgbm.predict(X_train)
test_preds_lgbm = lgbm.predict(X_test)

# Evaluation

In [ ]:
print("LGBM Train Accuracy:", lgbm.score(X_train, y_train))
print("LGBM Test Accuracy:", lgbm.score(X_test, y_test))

# Data Preprocessing

In [ ]:
def clean_math_text_final(text):
    text = str(text)
    text = re.sub(r'^\s*\d+\.\s*', '', text)    
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'#\w+', ' ', text)
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()

    return text

# Re-run both models on Cleaned Data

In [ ]:
train['Question'] = train['Question'].apply(clean_math_text_final)
test['Question'] = test['Question'].apply(clean_math_text_final)

train.head()

In [ ]:
X = train["Question"]
y = train["label"]
display(X.shape, y.shape, X.head(2), y.head(2))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
lr = Pipeline([('vect', CountVectorizer()),
               ('tfidf', TfidfTransformer()),
               ('clf', LogisticRegression(class_weight="balanced", multi_class='ovr'))
              ])
lr.fit(X_train, y_train)

In [ ]:
train_preds_lr = lr.predict(X_train)
test_preds_lr = lr.predict(X_test)

In [ ]:
print("Logistic Regression Train Accuracy:", lr.score(X_train, y_train))
print("Logistic Regression Test Accuracy:", lr.score(X_test, y_test))

In [ ]:
lgbm = Pipeline([('vect', CountVectorizer()),
               ('tfidf', TfidfTransformer()),
               ('clf', LGBMClassifier(is_unbalance = True, verbose = -1))
              ])

lgbm.fit(X_train, y_train)

In [ ]:
train_preds_lgbm = lgbm.predict(X_train)
test_preds_lgbm = lgbm.predict(X_test)

In [ ]:
print("LGBM Train Accuracy:", lgbm.score(X_train, y_train))
print("LGBM Test Accuracy:", lgbm.score(X_test, y_test))

# SMOTE

In [ ]:
print("\n--- Count Vectorization ---")
count_vectorizer = CountVectorizer()

count_vectorizer.fit(X_train)

X_train_counts = count_vectorizer.transform(X_train)
X_test_counts = count_vectorizer.transform(X_test)
print("Shape of Count Vectorizer train features:", X_train_counts.shape)
print("Shape of Count Vectorizer test features:", X_test_counts.shape)


print("\n--- TF-IDF Transformation ---")
tfidf_transformer = TfidfTransformer()

tfidf_transformer.fit(X_train_counts)

X_train_tfidf = tfidf_transformer.transform(X_train_counts)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)
print("Shape of TF-IDF train features:", X_train_tfidf.shape)
print("Shape of TF-IDF test features:", X_test_tfidf.shape)

print("\n--- SMOTE Oversampling ---")
print("Shape of TF-IDF train features before SMOTE:", X_train_tfidf.shape)
print("Original training labels:", Counter(y_train))

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_tfidf, y_train)

print("Shape of TF-IDF train features AFTER SMOTE:", X_train_resampled.shape)
print("Labels in resampled training set:", Counter(y_train_resampled))

print("\n--- Logistic Regression Training ---")
lr_model = LogisticRegression(multi_class='ovr')

lr_model.fit(X_train_resampled, y_train_resampled)
print("Logistic Regression model trained.")

print("\n--- Evaluation ---")

y_pred_train = lr_model.predict(X_train_resampled)
y_pred_test = lr_model.predict(X_test_tfidf)


train_f1_micro = f1_score(y_train_resampled, y_pred_train, average='micro')
test_f1_micro = f1_score(y_test, y_pred_test, average='micro')

print(f"Logistic Regression Train F1 Score (Macro): {train_f1_micro:.4f}")
print(f"Logistic Regression Test F1 Score (Macro): {test_f1_micro:.4f}")

In [ ]:
print("\n--- LGBM Training ---")
lgbm = LGBMClassifier(verbose = -1)
lgbm.fit(X_train_resampled, y_train_resampled)
print("LGBM model trained.")

print("\n--- Evaluation ---")

y_pred_train = lgbm.predict(X_train_resampled)
y_pred_test = lgbm.predict(X_test_tfidf)


train_f1_micro = f1_score(y_train_resampled, y_pred_train, average='micro')
test_f1_micro = f1_score(y_test, y_pred_test, average='micro')

print(f"LGBM Train F1 Score (Macro): {train_f1_micro:.4f}")
print(f"LGBM Test F1 Score (Macro): {test_f1_micro:.4f}")

# Kaggle Competition Submission

In [ ]:
test_counts = count_vectorizer.transform(test["Question"])
test_tfidf = tfidf_transformer.transform(test_counts)

predicted_labels = lgbm.predict(test_tfidf)

submission_df = pd.DataFrame({
    'id': test['id'],
    'label': predicted_labels
})


submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"Submission file '{submission_filename}' created successfully.")
display(submission_df.head())